# PySpark Week 6 Assignment

In [12]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .master("local[*]") \
    .getOrCreate()

In [14]:
# Read CSV file
import os
from pyspark.sql.functions import col

csv_path = r"/content/Sample - Superstore.csv"
df = spark.read.csv(csv_path, header=True, inferSchema=True)

df

DataFrame[Row ID: int, Order ID: string, Order Date: string, Ship Date: string, Ship Mode: string, Customer ID: string, Customer Name: string, Segment: string, Country: string, City: string, State: string, Postal Code: int, Region: string, Product ID: string, Category: string, Sub-Category: string, Product Name: string, Sales: string, Quantity: string, Discount: string, Profit: double]

In [15]:
# Print the schema
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [16]:
from pyspark.sql.functions import col

# Keep only valid numeric Sales values
df = df.filter(
    col("Sales").rlike("^[0-9.]+$")
)

# Convert Sales to double
df = df.withColumn(
    "Sales",
    col("Sales").cast("double")
)

# Show updated schema
df.printSchema()

# Display the data
df.show(5)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+------

In [11]:
from pyspark.sql.functions import col

# Check if the Sales column contains any invalid (non-numeric) values
invalid_sales = df.filter(
    ~col("Sales").rlike("^[0-9.]+$")
)

# Display the invalid rows
invalid_sales.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+---------------+-------------+--------+------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name    |Segment    |Country      |City         |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                                     |Sales          |Quantity     |Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+---------------+-------------+------

In [17]:
# Save the DataFrame as a Parquet file
df.write.mode("overwrite").parquet("parquet_data")

# Read the Parquet file
parquet_df = spark.read.parquet("parquet_data")

# Display the first 5 rows
parquet_df.show(5)

# Print the schema
parquet_df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [18]:
# Select required columns
selected_df = df.select(
    "Order ID",
    "Customer Name",
    "Category",
    "Sales",
    "Profit"
)

# Display the selected columns
selected_df.show(5)

+--------------+---------------+---------------+--------+--------+
|      Order ID|  Customer Name|       Category|   Sales|  Profit|
+--------------+---------------+---------------+--------+--------+
|CA-2016-152156|    Claire Gute|      Furniture|  261.96| 41.9136|
|CA-2016-152156|    Claire Gute|      Furniture|  731.94| 219.582|
|CA-2016-138688|Darrin Van Huff|Office Supplies|   14.62|  6.8714|
|US-2015-108966| Sean O'Donnell|      Furniture|957.5775|-383.031|
|US-2015-108966| Sean O'Donnell|Office Supplies|  22.368|  2.5164|
+--------------+---------------+---------------+--------+--------+
only showing top 5 rows


In [19]:
from pyspark.sql.functions import col

# Filter records where Region is West and Sales is greater than 500
filtered_df = df.filter(
    (col("Region") == "West") &
    (col("Sales") > 500)
)

# Display the filtered data
filtered_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     8|CA-2014-115812|06-09-2014| 6/14/2014|Standard Class|   BH-11710|Brosina Hoffman| Consumer|United States|Los Angeles|California|      90032|  West|TEC-PH-10002275|     Technology|      Phones|Mitel 5320 IP Pho...| 907.152|

In [20]:
# Rename the Sales column
renamed_df = df.withColumnRenamed(
    "Sales",
    "TotalSales"
)

# Display the result
renamed_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|TotalSales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Som

In [21]:
#cast datatypes
from pyspark.sql.functions import col

# Convert Quantity to Integer
cast_df = renamed_df.withColumn(
    "Quantity",
    col("Quantity").cast("int")
)

# Convert Discount to Double
cast_df = cast_df.withColumn(
    "Discount",
    col("Discount").cast("double")
)

# Print the schema
cast_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- TotalSales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [22]:
from pyspark.sql.functions import col

# Add a new column with 18% tax
new_df = cast_df.withColumn(
    "FinalSales",
    col("TotalSales") * 1.18
)

# Display the result
new_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|TotalSales|Quantity|Discount|  Profit|        FinalSales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+------------------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| So

In [23]:
# Remove rows with null values
drop_null_df = new_df.na.drop()

# Fill null values with 0
fill_null_df = new_df.na.fill(0)

# Display the result
fill_null_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|TotalSales|Quantity|Discount|  Profit|        FinalSales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+----------+--------+--------+--------+------------------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| So

In [24]:
# Select required columns
transform_df = fill_null_df.select(
    "Order ID",
    "Region",
    "Category",
    "FinalSales"
)

# Filter records
transform_df = transform_df.filter(
    col("FinalSales") > 500
)

# Rename a column
transform_df = transform_df.withColumnRenamed(
    "FinalSales",
    "NetSales"
)

# Display the result
transform_df.show(5)

+--------------+------+----------+----------+
|      Order ID|Region|  Category|  NetSales|
+--------------+------+----------+----------+
|CA-2016-152156| South| Furniture|  863.6892|
|US-2015-108966| South| Furniture|1129.94145|
|CA-2014-115812|  West|Technology|1070.43936|
|CA-2014-115812|  West| Furniture|2013.29712|
|CA-2014-115812|  West|Technology|1075.48032|
+--------------+------+----------+----------+
only showing top 5 rows


In [25]:
# Display records
transform_df.show(5)

# Count total rows
print("Total Rows:", transform_df.count())

# Display schema
transform_df.printSchema()

+--------------+------+----------+----------+
|      Order ID|Region|  Category|  NetSales|
+--------------+------+----------+----------+
|CA-2016-152156| South| Furniture|  863.6892|
|US-2015-108966| South| Furniture|1129.94145|
|CA-2014-115812|  West|Technology|1070.43936|
|CA-2014-115812|  West| Furniture|2013.29712|
|CA-2014-115812|  West|Technology|1075.48032|
+--------------+------+----------+----------+
only showing top 5 rows
Total Rows: 1385
root
 |-- Order ID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- NetSales: double (nullable = false)



In [26]:
#transfromation
from pyspark.sql.functions import sum

# Calculate total sales by Region
group_df = fill_null_df.groupBy("Region").agg(
    sum("FinalSales").alias("TotalSales")
)

# Display the result
group_df.show()

+-------+-----------------+
| Region|       TotalSales|
+-------+-----------------+
|  South|459000.6303000005|
|Central|587405.0299040014|
|   East| 793188.983719998|
|   West|841896.1865099999|
+-------+-----------------+



In [27]:
# Shuffle occurs during groupBy operation
shuffle_df = fill_null_df.groupBy("Category").count()

# Display the result
shuffle_df.show()

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies| 5781|
|      Furniture| 2074|
|     Technology| 1839|
+---------------+-----+



In [28]:
# Read the Parquet file
parquet_df = spark.read.parquet("parquet_data")

# Filter records
pushdown_df = parquet_df.filter(
    col("Region") == "West"
)

# Display the result
pushdown_df.show(5)

+------+--------------+----------+---------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+---------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|     3|CA-2016-138688|06-12-2016|6/16/2016|  Second Class|   DV-13045|Darrin Van Huff|Corporate|United States|Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|  14.62|       2|  

In [29]:
# Read CSV file
csv_df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

# Read Parquet file
parquet_df = spark.read.parquet("parquet_data")

# Display sample records
csv_df.show(5)
parquet_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [30]:
#Build Complete Data Pipeline
from pyspark.sql.functions import col

# Read CSV file
pipeline_df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

# Filter valid Sales values
pipeline_df = pipeline_df.filter(
    col("Sales").rlike("^[0-9.]+$")
)

# Convert Sales to double
pipeline_df = pipeline_df.withColumn(
    "Sales",
    col("Sales").cast("double")
)

# Filter required records
pipeline_df = pipeline_df.filter(
    col("Sales") > 500
)

# Select required columns
pipeline_df = pipeline_df.select(
    "Order ID",
    "Customer Name",
    "Region",
    "Sales"
)

# Display the result
pipeline_df.show(5)


+--------------+---------------+------+--------+
|      Order ID|  Customer Name|Region|   Sales|
+--------------+---------------+------+--------+
|CA-2016-152156|    Claire Gute| South|  731.94|
|US-2015-108966| Sean O'Donnell| South|957.5775|
|CA-2014-115812|Brosina Hoffman|  West| 907.152|
|CA-2014-115812|Brosina Hoffman|  West|1706.184|
|CA-2014-115812|Brosina Hoffman|  West| 911.424|
+--------------+---------------+------+--------+
only showing top 5 rows


In [31]:
# Save the processed data as CSV
pipeline_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

print("CSV file saved successfully.")

CSV file saved successfully.


In [32]:
# Save the processed data as Parquet
pipeline_df.write \
    .mode("overwrite") \
    .parquet("output_parquet")

print("Parquet file saved successfully.")

Parquet file saved successfully.


In [33]:
# Display only the first 5 rows
pipeline_df.show(5)

# Count total records
print("Total Records:", pipeline_df.count())

# Avoid using collect() on large datasets
# collect() loads all data into the driver memory

+--------------+---------------+------+--------+
|      Order ID|  Customer Name|Region|   Sales|
+--------------+---------------+------+--------+
|CA-2016-152156|    Claire Gute| South|  731.94|
|US-2015-108966| Sean O'Donnell| South|957.5775|
|CA-2014-115812|Brosina Hoffman|  West| 907.152|
|CA-2014-115812|Brosina Hoffman|  West|1706.184|
|CA-2014-115812|Brosina Hoffman|  West| 911.424|
+--------------+---------------+------+--------+
only showing top 5 rows
Total Records: 1151


## Spark Concepts and Best Practices

- Spark Architecture: The driver program coordinates the job, the cluster manager allocates resources, and executors run tasks on worker nodes.
- Lazy Evaluation: Spark does not execute transformations immediately; it builds a plan and executes only when an action is called.
- DAG (Directed Acyclic Graph): Spark creates a DAG of stages to show the dependency between transformations.
- Transformations vs Actions: Transformations create new DataFrames without immediate execution, while actions trigger execution and return results.
- Shuffle: A shuffle redistributes data across partitions, which can be expensive.
- Predicate Pushdown: Spark can push filters closer to the data source to reduce the amount of data read.
- CSV vs Parquet: CSV is simple and human-readable, while Parquet is columnar, compressed, and better for analytics.
- Client Mode vs Cluster Mode: In client mode, the driver runs on the machine that submitted the job; in cluster mode, the driver runs on the cluster.
- Best practices for large datasets: Use Parquet, select only needed columns, filter early, avoid unnecessary shuffles, and cache data when reused.